In [0]:
df = spark.read.table("samples.bakehouse.sales_transactions")
display(df)

In [0]:
df.count()

In [0]:
# df.first()
df.tail(1)

In [0]:
from pyspark.sql.functions import (min, max, col)

df2 = df.select(min(col("quantity")).alias("min"), max(col("quantity")).alias("max"))
display(df2)

In [0]:
from pyspark.sql.functions import (sum, avg, col)

df2 = df.select(sum(col("quantity")).alias("sum"), avg(col("quantity")).alias("avg"))
display(df2)

In [0]:
%sql
select 
  min(quantity) as min,
  max(quantity) as max,
  sum(quantity) as sum,
  avg(quantity) as avg
from samples.bakehouse.sales_transactions

In [0]:
from pyspark.sql.functions import count, col

df.groupby("paymentMethod").agg(count(col("paymentMethod")).alias("count")).display()

## OR
# df.groupby("paymentMethod").count().display()

In [0]:
%sql
select paymentMethod, count(paymentMethod) as count
from samples.bakehouse.sales_transactions
group by paymentMethod;

In [0]:
from pyspark.sql.functions import count, col

df.groupby("paymentMethod").count().filter(col("count") > 1000).display()

In [0]:
from pyspark.sql.functions import collect_set, collect_list

df.groupby("paymentMethod").agg(collect_set("product"), collect_list("product")).display()

In [0]:
%sql
select paymentMethod, 
collect_set(product),
collect_list(product)
from samples.bakehouse.sales_transactions
group by paymentMethod;

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy("product").orderBy(col("quantity").desc())
df2 = df.withColumn("row_number_over_product", row_number().over(window_spec))
df2.select("product", "customerID", "quantity", "row_number_over_product").display()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, rank, dense_rank

window_spec = Window.partitionBy("product").orderBy(col("quantity").desc())
df2 = df.withColumn("row_number_over_product", row_number().over(window_spec))\
    .withColumn("rank_over_product", rank().over(window_spec))\
    .withColumn("dense_rank_over_product", dense_rank().over(window_spec)).display()

In [0]:
%sql
SELECT
  product,
  customerID,
  quantity,
  ROW_NUMBER() OVER (PARTITION BY product ORDER BY quantity DESC) AS row_number_over_product,
  RANK() OVER (PARTITION BY product ORDER BY quantity DESC) AS rank_over_product,
  DENSE_RANK() OVER (PARTITION BY product ORDER BY quantity DESC) AS dense_rank_over_product
FROM samples.bakehouse.sales_transactions